## 15.1. Spot detection

- Input: anchor image from 4 rounds <br>
- flow: MIP: from 4 -> 1 anchor image => apply filter => detect spot by trackpy => filtering by size <br>
- Output: csv file contain: name fov, spot ID, y,x (spot coordinate)

In [7]:
import glob
import trackpy as tp
import numpy as np
import pandas as pd
import skimage as ski
from skimage import io, filters
import matplotlib.pyplot as plt
from skimage.filters import gaussian
from skimage.morphology import white_tophat, disk
import scipy

In [2]:
PATH = '/Users/chann/Downloads/F1 Systems bio/Project/code/realistic_data/selected-tiles'

### 1. MIP and image processing

In [3]:
# Stack anchor images into 1 - MIP
def stack_image(folder_path, fov):
    anchor_files = sorted(glob.glob(f'{folder_path}/out_opt_flow_registered_{fov}_c*_Atto_490LS.tif'))
    anchor_list = [ski.io.imread(file) for file in anchor_files]
    anchor_stack = np.stack(anchor_list,axis = 0)
    anchor_combined = np.max(anchor_stack, axis=0)
    return anchor_combined

# preprocessing stacked image
def pre_image(im_stacked):
    im_stacked = im_stacked.astype(float)
    im_stacked_substraction = white_tophat(im_stacked)
    im_stacked_gauss = gaussian(im_stacked_substraction, sigma=0.8)
    return im_stacked_gauss


In [4]:
# Take all of fov
path_fov = sorted(glob.glob(f'{PATH}/out_opt_flow_registered_*_c01_DAPI.tif'))
name_fov = [path.split("registered_")[1].split("_c01")[0] for path in path_fov]#List comprehension
print(len(name_fov))

190


In [5]:
# calling function

anchor_stack_list = []
for name in name_fov:
    anchor_stack = stack_image(PATH,name)
    anchor_stack_list.append(anchor_stack) #output: list of anchor_MIP

anchor_stack_process = []
for anchor in anchor_stack_list:
    anchor_process = pre_image(anchor)
    anchor_stack_process.append(anchor_process)

print(len(anchor_stack_process)) #output: anchor_MIP_processed


190


### 2. Spot detection by trackpy

In [6]:
# Detect spot by trackpy

spot_dict = {}
for name, anchor in zip(name_fov, anchor_stack_process):
    spot_dict[name] = tp.locate(anchor, diameter = 5, percentile = 0, separation = 2) #output:dataframe

all_spots = pd.concat(spot_dict, names=["FOV", "Spot_ID"]) #concat many dataframe -> one df
all_spots_filtered = all_spots[(all_spots["mass"] > 50)].reset_index() #reset_index: from unflat -> flat

print(all_spots_filtered.head())


       FOV  Spot_ID         y           x        mass      size       ecc  \
0  X10_Y10        4  2.686869    1.474747   63.640381  1.087115  0.292884   
1  X10_Y10        5  4.289941  266.218935  108.638631  1.196642  0.105433   
2  X10_Y10        6  1.991736  868.181818   77.782688  1.071802  0.228903   
3  X10_Y10        9  3.809717  617.036437  158.779537  1.246858  0.097704   
4  X10_Y10       14  5.926829  434.170732   52.712235  1.168697  0.062392   

      signal    raw_mass        ep  
0  15.427971  541.459352  0.009552  
1  17.999300  568.406411  0.009089  
2  15.427971  476.631756  0.010889  
3  23.141957  900.767189  0.005684  
4   8.999650  293.063094  0.018037  


### 3. Save as csv

In [7]:
# Save spot coordinate
df_all_spots = pd.DataFrame({
    "FOV": all_spots_filtered["FOV"].values,
    "Spot_ID": all_spots_filtered.groupby("FOV").cumcount().values + 1, #numbers each item inside a group from -> size of that group -1
    "y": all_spots_filtered["y"].values,
    "x": all_spots_filtered["x"].values,
})

df_all_spots.to_csv("/Users/chann/Downloads/F1 Systems bio/Project/code/Results/Intermediates/Decoding_Spot detection/spots_detect_all_fov.csv", index=False)
print(df_all_spots.head())

       FOV  Spot_ID         y           x
0  X10_Y10        1  2.686869    1.474747
1  X10_Y10        2  4.289941  266.218935
2  X10_Y10        3  1.991736  868.181818
3  X10_Y10        4  3.809717  617.036437
4  X10_Y10        5  5.926829  434.170732


Groupby:  splits data into groups based on specific keys or columns. in this case -> split follow FOV <br>
.values: take numpy array from col/series, cut index (only remain values)

## 15.2. Decoding

- Input: 4 channels <br>
- Flow: stack 4 images together but NOT MIP => got 4D matrix (round, channel, y,x) => preprocessing(normalization(percentilecliping, minmax(0,1), histogram matching) => decoding <br>
- Output: barcode => gene name + efficiency

### 1. Stack channels -> 4D

In [8]:
# Take all of fov
path_fov = sorted(glob.glob(f'{PATH}/out_opt_flow_registered_*_c01_DAPI.tif'))
name_fov_list = [path.split("registered_")[1].split("_c01")[0] for path in path_fov]  #List comprehension
print(len(name_fov))

190


In [9]:
channel_name = ['Atto_425', 'Alexa_488', 'Alexa_568', 'Alexa_647']
channel_number = len(channel_name)

round_name = ['c01', 'c02', 'c03', 'c04']
round_number = len(round_name)

height, width = 1000, 1000

#Read image
def read_image(fov, round, channel):
    file_path = f"{PATH}/out_opt_flow_registered_{fov}_{round}_{channel}.tif"
    return io.imread(file_path)

# Stack image
stack_channel_dict = {}
for fov_name in name_fov_list:
    stack_image = np.zeros((round_number, channel_number, height, width),dtype = np.uint16)
    for r_number, r_name in enumerate(round_name):
        for c_number, c_name in enumerate(channel_name):
            image = read_image(fov_name, r_name, c_name)
            stack_image[r_number,c_number] = image

    stack_channel_dict[fov_name] = stack_image

print(len(stack_channel_dict), stack_image.dtype, stack_image.shape)


190 uint16 (4, 4, 1000, 1000)


Theoretically, we can decode directly, but we need to normalize the intensity of channels -> make sure that dont have bias: <br>
- percentile clipping (because have outlier) + normalization (0,1) -> no basement <br>
- histogram matching: no dominate channels across a round ->

### 2. Preprocessing

Because we have outlier => need to do percentile clipping before normalization (0,1)

#### 2.1. Percentile clipping and normalization


In [10]:
# 1 image: Percentile clipping + normalization (0,1)

def percentile_clip(channel, lower_percent, upper_percent):
    lower = np.percentile(channel, lower_percent)
    upper = np.percentile(channel, upper_percent)
    clipped = np.clip(channel, lower, upper)
    normalize  = (clipped - lower)/(upper - lower)
    normalized = np.clip(normalize,0,1)
    return lower, upper, normalized

# 16 images

def all_percentile (stack_image, lower_percentile, upper_percentile):
    stack_normalized = np.zeros_like(stack_image,dtype=np.float32)
    for r in range(round_number):
        for c in range(channel_number):
            channel = stack_image[r,c]
            _, _, normalized = percentile_clip(channel, lower_percentile, upper_percentile)
            stack_normalized[r, c] = normalized
    return stack_normalized #

# all fov
stack_channel_normalized ={}
for fov_name, stack_image in stack_channel_dict.items(): #dict.items => tuple(key,value)
    stack_channel_normalized[fov_name] = all_percentile(stack_image,0,99.9)

#### Decoding all fov ( percentile clipping )

In [16]:
# decode: 1 fov function
base_letters = ['A', 'C', 'G', 'T']


def decode_percentile(image_percentile_normalized, spots_coordinates_fov):
    """ Decode 1 fov: for each spot coordinate (y,x): take intensities through 4 channels => take max

    Args:
        image_percentile_normalized (array): all the matched images in this fov
        spots_coordinates_fov(array): spots coordinates in this fov

    Return:
        decoded sequences"""
    n_round = image_percentile_normalized.shape[0]
    decoded_sequences_percentile = []

    for y, x in spots_coordinates_fov:
        y, x = int(round(y)), int(round(x))
        sequence_percentile = ""
        for r in range(n_round):
            values = image_percentile_normalized[r, :, y, x]  #take intensities of this spot coordinate across all 4 channels
            index_max = np.argmax(values)  #choose the maximum one
            base = base_letters[index_max]
            sequence_percentile += base
        decoded_sequences_percentile.append(sequence_percentile)

    return decoded_sequences_percentile

In [30]:
#Loop all fov
all_fov_decode_percentile = {}
for fov_name, percentile_image in stack_channel_normalized.items():
    df_spot_fov = df_all_spots[
        df_all_spots['FOV'] == fov_name]  #output: all spot coordinate of this fov (satisfied name condition)
    spots_coordinates = df_spot_fov[['y', 'x']].to_numpy()  #from dataframe to numpy

    all_fov_decode_percentile[fov_name] = decode_percentile(percentile_image, spots_coordinates)
    #Output: dictionary (fov: barcode)

In [31]:
# Checking step: the number of detected spots should = number of decoded sequence
for fov_name, seqs in all_fov_decode_percentile.items():
    n_spots = (df_all_spots['FOV'] == fov_name).sum()
    print(fov_name, len(seqs), n_spots)

X10_Y10 1112 1112
X10_Y11 1761 1761
X10_Y12 2046 2046
X10_Y13 2094 2094
X10_Y14 1907 1907
X10_Y15 2278 2278
X10_Y2 631 631
X10_Y3 752 752
X10_Y4 1110 1110
X10_Y5 1368 1368
X10_Y6 1589 1589
X10_Y7 1478 1478
X10_Y8 1369 1369
X10_Y9 1403 1403
X11_Y10 462 462
X11_Y11 1739 1739
X11_Y12 2199 2199
X11_Y13 1919 1919
X11_Y14 2387 2387
X11_Y15 2807 2807
X11_Y1 450 450
X11_Y2 1173 1173
X11_Y3 1248 1248
X11_Y4 988 988
X11_Y5 1127 1127
X11_Y6 810 810
X11_Y7 993 993
X11_Y8 1085 1085
X11_Y9 1014 1014
X12_Y0 313 313
X12_Y10 565 565
X12_Y11 612 612
X12_Y12 461 461
X12_Y13 542 542
X12_Y14 1269 1269
X12_Y15 1558 1558
X12_Y1 1297 1297
X12_Y2 1090 1090
X12_Y3 1014 1014
X12_Y4 1262 1262
X12_Y5 1218 1218
X12_Y6 1270 1270
X12_Y7 1463 1463
X12_Y8 1493 1493
X12_Y9 1252 1252
X13_Y0 362 362
X13_Y10 1483 1483
X13_Y11 2311 2311
X13_Y12 2497 2497
X13_Y13 2393 2393
X13_Y14 2200 2200
X13_Y15 1037 1037
X13_Y1 1409 1409
X13_Y2 1159 1159
X13_Y3 1027 1027
X13_Y4 1297 1297
X13_Y5 1418 1418
X13_Y6 1238 1238
X13_Y7 1677 1677

In [32]:
#Save barcode into the dataframe of spot coordinate

df_all_spots['barcode'] = None  # add a new column

for fov_name, barcodes in all_fov_decode_percentile.items():
    mask = df_all_spots['FOV'] == fov_name  #create a mask column (True/False): true if fov_name == FOV
    df_all_spots.loc[mask, 'barcode'] = barcodes  #choose the row have "True"

In [33]:
# Map codebook
# read code book
codebook_df = pd.read_csv("/Users/chann/Downloads/F1 Systems bio/Project/code/realistic_data/taglist.csv")
# Mapping to codebook

seq_to_gene = dict(zip(codebook_df['Code'], codebook_df['Name']))  #create dictionary from codebook (key: barcode, value: gene name)

result_decode_percentile_df = df_all_spots.copy()
result_decode_percentile_df['gene'] = result_decode_percentile_df['barcode'].map(seq_to_gene).fillna(
    'invalid')  #mapping barcode. if cannot find -> NaN
result_decode_percentile_df['valid'] = result_decode_percentile_df['gene'] != 'invalid'  #NaN -> invalid

print(result_decode_percentile_df.head(20))

        FOV  Spot_ID          y           x barcode     gene  valid
0   X10_Y10        1   2.686869    1.474747    TACG    Gapdh   True
1   X10_Y10        2   4.289941  266.218935    TACA  invalid  False
2   X10_Y10        3   1.991736  868.181818    TACA  invalid  False
3   X10_Y10        4   3.809717  617.036437    TACG    Gapdh   True
4   X10_Y10        5   5.926829  434.170732    ATAA  invalid  False
5   X10_Y10        6   6.906977  709.302326    AAGA  invalid  False
6   X10_Y10        7   9.372549  275.366013    TTGC   Atp2b4   True
7   X10_Y10        8   9.687179  334.302564    TACG    Gapdh   True
8   X10_Y10        9   8.093960  344.416107    ACTC   Grin3a   True
9   X10_Y10       10  11.200000  527.683721    AAAG  invalid  False
10  X10_Y10       11   9.712418  356.588235    AAAA  invalid  False
11  X10_Y10       12  12.123288  636.328767    TACG    Gapdh   True
12  X10_Y10       13  11.755319  426.351064    TACA  invalid  False
13  X10_Y10       14  13.983333  642.373810    T

In [34]:
print(result_decode_percentile_df['valid'].mean())                  # tổng thể
print(result_decode_percentile_df.groupby('FOV')['valid'].mean())

0.617230461589735
FOV
X10_Y10    0.571942
X10_Y11    0.630892
X10_Y12    0.679863
X10_Y13    0.546800
X10_Y14    0.629785
             ...   
X23_Y4     0.333333
X23_Y5     0.429630
X23_Y6     0.431034
X23_Y7     0.525000
X23_Y8     0.520325
Name: valid, Length: 190, dtype: float64


#### 2.2. Histogram matching by Starfish

#### Slice - add z - Create initial Imagestack

In [9]:
import numpy as np
import starfish
from starfish import ImageStack
from starfish.types import Axes

In [10]:
print(starfish.__version__)
print(trackpy.__version__)


0.4.0


NameError: name 'trackpy' is not defined

In [12]:
# 1 round: match histogram
def match_histogram_one_round (image_stacked_1_fov, r):
    """ Match histogram all channels (4 channels) across 1 round.

     Arg:
        image_stacked_1_fov (array): 16 stacked images in 1 fov
        r: round number: to slice round.

    Returns: 4 images per round (matched)."""
    round_ch = image_stacked_1_fov[r:r+1]. astype(np.float32) #all of image stacked in one fov; r: round index
    round_ch_5d = round_ch[:,:,np.newaxis,:,:]

    round_stack = ImageStack.from_numpy(round_ch_5d)
    mh_round_ch = starfish.image.Filter.MatchHistograms({Axes.CH})
    scaled_round_ch = mh_round_ch.run(round_stack, in_place = False, verbose = False)
    return scaled_round_ch


input: 16 images stacked in one fov  => slice 1 round => add z axis => start imagestack5d => start match histogram in starfish

In [13]:
# 4 round + change datatype
def match_histogram_four_round (image_stacked_1_fov):
    n_rounds = image_stacked_1_fov.shape[0]
    result_arrays = []
    for r_number in range(n_rounds):
        scaled = match_histogram_one_round(image_stacked_1_fov,r_number) #match histogram for round r -> imagestack (1,4,1,y,x)
        array_5d = scaled.xarray.values # take numpy value out (1,4,1,y,x): syntax: .x.array.values (access and take value -> numpy array)
        array_3d = array_5d [0, :, 0, :, :] # - round and z (4,y,x)
        result_arrays.append(array_3d) # add in list
    return np.stack(result_arrays) #arrange 4 round together (4,4,y,x) =>4D numpy. axis = 0/1/2/3... => determine position of new axis != np.concat


In [14]:
from functools import partialmethod
from tqdm import tqdm

tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)

In [15]:
#all fov
all_fov_matching= {}
for fov_name, stack_image_normalize in stack_channel_normalized.items():
    all_fov_matching[fov_name] = match_histogram_four_round(stack_image_normalize)
    


KeyboardInterrupt: 

Flow of Histogram matching: <br>
- For each round => match channels across 1 round -> return matched histogram - imagestack of starfish (5d) (4) <br>
- For all of rounds => replicate one round's function => loop over rounds => list of matched histogram => change the datatype (5 => 3d) => return: numpy array (4d) (4 elements - 4 rounds)

In [ ]:
# decode: 1 fov function
base_letters = ['A', 'C', 'G', 'T']

def decode(image_matched_4d, spots_coordinates_fov):
    """ Decode 1 fov: for each spot coordinate (y,x): take intensities through 4 channels => take max

    Args:
        image_matched_4d(array): all the matched images in this fov
        spots_coordinates_fov(array): spots coordinates in this fov

    Return:
        decoded sequences"""
    n_round = image_matched_4d.shape[0]
    decoded_sequences = []

    for y, x in spots_coordinates_fov:
        y, x = int(round(y)), int(round(x))
        sequence = ""
        for r in range(n_round):
            values = image_matched_4d[r, :, y, x]  #take intensities of this spot coordinate across all 4 channels
            index_max = np.argmax(values) #choose the maximum one
            base = base_letters[index_max]
            sequence += base
        decoded_sequences.append(sequence)

    return decoded_sequences

In [ ]:
#Loop all fov
all_fov_decode = {}
for fov_name, matched_image in all_fov_matching.items():
    df_spot_fov = df_all_spots[df_all_spots['FOV'] == fov_name] #output: all spot coordinate of this fov (satisfied name condition)
    spots_coordinates = df_spot_fov[['y', 'x']].to_numpy() #from dataframe to numpy

    all_fov_decode[fov_name] = decode(matched_image, spots_coordinates)
    #Output: dictionary (fov: barcode)

In [ ]:
# Checking step: the number of detected spots should = number of decoded sequence
for fov_name, seqs in all_fov_decode.items():
    n_spots = (df_all_spots['FOV'] == fov_name).sum()
    print(fov_name, len(seqs), n_spots)

In [ ]:
#Save barcode into the dataframe of spot coordinate

df_all_spots['barcode'] = None # add a new column

for fov_name, barcodes in all_fov_decode.items():
    mask = df_all_spots['FOV'] == fov_name #create a mask column (True/False): true if fov_name == FOV
    df_all_spots.loc[mask, 'barcode'] = barcodes #choose the row have "True"

In [ ]:
# Map codebook
# read code book
codebook_df = pd.read_csv("/Users/chann/Downloads/F1 Systems bio/Project/code/realistic_data/taglist.csv")

In [ ]:
# Mapping to codebook

seq_to_gene = dict(zip(codebook_df['Code'], codebook_df['Name'])) #create dictionary from codebook (key: barcode, value: gene name)

result_decode_df = df_all_spots.copy()
result_decode_df['gene'] = result_decode_df['barcode'].map(seq_to_gene).fillna('invalid') #mapping barcode. if cannot find -> NaN
result_decode_df['valid'] = result_decode_df['gene'] != 'invalid' #NaN -> invalid

print(result_decode_df.head(20))

In [ ]:
print(result_decode_df['valid'].mean())                  # tổng thể
print(result_decode_df.groupby('FOV')['valid'].mean())   # từng FOV

In [ ]:
#Save as csv1
predictions_df = result_decode_df[['FOV', 'x', 'y', 'barcode', 'gene']].copy()
predictions_df.columns = ['fov', 'x', 'y', 'barcode', 'gene']

predictions_df.to_csv("decode_predictions.csv", index=False)
print(predictions_df.head())

In [ ]:
#Save as csv2

counts_df = (
    result_decode_df[result_decode_df['valid']]      # take spot invalid out
    .groupby(['FOV', 'gene'])
    .size()
    .reset_index(name='count')
)
counts_df.columns = ['fov', 'gene', 'count']          # change name of column

counts_df.to_csv("/Users/chann/Downloads/F1 Systems bio/Project/code/Results/Finals/gene_counts.csv", index=False)
print(counts_df.head())